In this notebook pobieramy 3 miesiace of broadcasts from radio ergo z ktorych bedziemy chcieli wyciagnac informacje o food insecurity. 

Okres dla ktorego pobierzemy dane to Jan-Mar 2022
uzasadnienie: w tym okresie sytuacja znacznie sie pograszyla i chcielibysmy zobaczyc czy w radiu w przeciagu tych trzech miesiecy pojawily sie informacje ze syutacja bedzie ulegac pogorszeniu. 

https://www.ipcinfo.org/ipc-country-analysis/details-map/en/c/1155438/?iso3=SOM

https://www.ipcinfo.org/ipc-country-analysis/details-map/en/c/1155523/?iso3=SOM

In [3]:
"""
SoundCloud Audio Downloader

Clean, efficient tool for downloading SoundCloud tracks by date range or specific URLs.
Optimized for notebook environments with proper error handling and logging.
"""

import os
import re
import time
import logging
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Optional, Tuple
from urllib.parse import urljoin

import yt_dlp
import requests
from bs4 import BeautifulSoup


# ============================================================================
# Path Resolution (Moved to top for logging)
# ============================================================================

def resolve_project_root() -> Path:
    """
    Resolve the project root directory.
    
    Returns:
        Path to project root (somali-radios-with-ai-for-food-security/)
        
    Raises:
        RuntimeError: If project root cannot be found
    """
    # Start from current working directory
    current = Path.cwd()
    
    # Check if we're already in the project directory
    if current.name == "somali-radios-with-ai-for-food-security":
        return current
    
    # Check common studio paths
    studio_path = Path("/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security")
    if studio_path.exists():
        return studio_path
    
    # Walk up from current directory looking for project folder
    for parent in [current] + list(current.parents):
        project_path = parent / "somali-radios-with-ai-for-food-security"
        if project_path.exists():
            return project_path
    
    raise RuntimeError(
        "Could not locate project root 'somali-radios-with-ai-for-food-security'. "
        f"Current directory: {current}"
    )

# ============================================================================
# Logging Configuration (Uses resolve_project_root)
# ============================================================================

# 1. Find project root
PROJECT_ROOT = resolve_project_root()

# 2. Create a 'logs' directory if it doesn't exist
LOG_DIR = PROJECT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# 3. Define the log file path
LOG_FILE_PATH = LOG_DIR / "soundcloud_downloader.log"

# 4. Configure logging
# Remove existing handlers to avoid duplicate logs if run in a notebook cell multiple times
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
    
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE_PATH), # <-- Fix 1: Log to logs/ directory
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

logger.info(f"Logging configured. Log file will be saved to: {LOG_FILE_PATH}")


# ============================================================================
# Constants
# ============================================================================

MONTH_NAMES = {
    'january': 1, 'february': 2, 'march': 3, 'april': 4, 'may': 5, 'june': 6,
    'july': 7, 'august': 8, 'september': 9, 'october': 10, 'november': 11, 'december': 12,
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'jun': 6, 'jul': 7,
    'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}

SOUNDCLOUD_URL_PATTERN = r'^https?://(?:www\.)?soundcloud\.com/[\w-]+/[\w-]+'

DATE_PATTERNS = [
    r'(\d{1,2})-([a-z]+)-(\d{4})',  # DD-month-YYYY or DD-mon-YYYY
    r'(\d{4})-(\d{1,2})-(\d{1,2})'  # YYYY-MM-DD
]

DEFAULT_USER_AGENT = 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'


# ============================================================================
# Path Resolution (Remaining function)
# ============================================================================

def get_raw_data_dir() -> Path:
    """
    Get the 01_raw data directory, creating it if needed.
    
    Returns:
        Path to data/01_raw/ directory
    """
    # We can use PROJECT_ROOT since it's already defined
    raw_dir = PROJECT_ROOT / "data" / "01_raw"
    raw_dir.mkdir(parents=True, exist_ok=True)
    return raw_dir


# ============================================================================
# URL Validation & Parsing
# ============================================================================

def validate_soundcloud_url(url: str) -> bool:
    """
    Validate if URL is a valid SoundCloud track URL.
    
    Args:
        url: URL to validate
        
    Returns:
        True if valid SoundCloud URL, False otherwise
    """
    return bool(re.match(SOUNDCLOUD_URL_PATTERN, url, re.IGNORECASE))


def extract_date_from_url(url: str) -> Optional[datetime]:
    """
    Extract date from SoundCloud URL using various patterns.
    
    Args:
        url: SoundCloud URL to parse
        
    Returns:
        datetime object if date found, None otherwise
    """
    url_path = url.split('/')[-1].lower()
    
    for pattern in DATE_PATTERNS:
        match = re.search(pattern, url_path, re.IGNORECASE)
        if not match or len(match.groups()) != 3:
            continue
            
        try:
            day, month_str, year = match.groups()
            
            # Convert month to integer
            if month_str.isdigit():
                month = int(month_str)
            else:
                month = MONTH_NAMES.get(month_str.lower())
            
            if month and 1 <= month <= 12:
                return datetime(int(year), month, int(day))
                
        except (ValueError, TypeError) as e:
            logger.debug(f"Date parsing failed for {url}: {e}")
            continue
    
    return None


def extract_username_from_url(profile_url: str) -> str:
    """
    Extract username from SoundCloud profile URL.
    
    Args:
        profile_url: SoundCloud profile URL
        
    Returns:
        Username string
    """
    return profile_url.rstrip('/').split('/')[-1]


# ============================================================================
# Web Scraping
# ============================================================================

def create_session() -> requests.Session:
    """
    Create configured requests session.
    
    Returns:
        Configured requests.Session object
    """
    session = requests.Session()
    session.headers.update({'User-Agent': DEFAULT_USER_AGENT})
    return session


def fetch_profile_tracks(profile_url: str, session: requests.Session) -> List[str]:
    """
    Fetch all track URLs from a SoundCloud profile page.
    
    Args:
        profile_url: SoundCloud profile URL
        session: Configured requests session
        
    Returns:
        List of unique track URLs found on the profile
    """
    try:
        response = session.get(profile_url, timeout=30)
        response.raise_for_status()
    except Exception as e:
        logger.error(f"Failed to fetch profile {profile_url}: {e}")
        return []
    
    soup = BeautifulSoup(response.text, 'html.parser')
    username = extract_username_from_url(profile_url)
    track_urls = []
    
    # Find all track links
    for link in soup.find_all('a', href=True):
        href = link['href']
        
        # Filter for track URLs (not playlists, has proper depth)
        if not (href.startswith('/') and username in href and 
                '/sets/' not in href and href.count('/') >= 2):
            continue
        
        full_url = urljoin('https://soundcloud.com', href)
        if validate_soundcloud_url(full_url):
            track_urls.append(full_url)
    
    # Remove duplicates while preserving order
    return list(dict.fromkeys(track_urls))


def generate_url_patterns(username: str, date: datetime) -> List[str]:
    """
    Generate potential URL patterns for a given date.
    
    Args:
        username: SoundCloud username
        date: Date to generate patterns for
        
    Returns:
        List of potential URLs to check
    """
    base_url = f"https://soundcloud.com/{username}"
    
    day = date.day
    month_full = date.strftime('%B').lower()
    month_abbr = date.strftime('%b').lower()
    year = date.year
    
    # Common patterns used by Radio Ergo and similar accounts
    patterns = [
        f"idaacadda-{day:02d}-{month_abbr}-{year}",
        f"idaacadda-{day}-{month_abbr}-{year}",
        f"idaacadda-{day:02d}-{month_full}-{year}",
        f"idaacadda-{day}-{month_full}-{year}",
        f"show-{day:02d}-{month_abbr}-{year}",
        f"broadcast-{day}-{month_abbr}-{year}"
    ]
    
    return [f"{base_url}/{pattern}" for pattern in patterns]


def check_url_exists(url: str, session: requests.Session) -> bool:
    """
    Check if a SoundCloud URL is a valid track page. (FIX 3)
    
    A simple HEAD request is unreliable as SoundCloud returns 200
    for many non-track pages. This does a GET and looks for
    a key element present on track pages.
    
    Args:
        url: URL to check
        session: Configured requests session
        
    Returns:
        True if URL appears to be a valid track page, False otherwise
    """
    try:
        response = session.get(url, timeout=10)
        if response.status_code != 200:
            return False
        
        # Check for content specific to a track page
        # <meta property="og:type" content="music.song">
        soup = BeautifulSoup(response.text, 'html.parser')
        meta_tag = soup.find('meta', attrs={'property': 'og:type', 'content': 'music.song'})
        
        if meta_tag:
            return True
        
        # Fallback check (less reliable, but useful)
        play_button = soup.find('a', class_=re.compile(r'playButton'))
        if play_button:
            return True
        
        logger.debug(f"URL {url} is 200 OK but not a valid track page.")
        return False
        
    except Exception as e:
        logger.debug(f"Failed to check URL {url}: {e}")
        return False


# ============================================================================
# URL Collection
# ============================================================================

def collect_urls_in_date_range(
    profile_url: str, 
    start_date: datetime, 
    end_date: datetime,
    session: requests.Session
) -> List[str]:
    """
    Collect all SoundCloud URLs within a date range.
    
    Combines two strategies:
    1. Parse existing tracks from profile page
    2. Generate and check potential URLs based on date patterns
    
    Args:
        profile_url: SoundCloud profile URL
        start_date: Start date (inclusive)
        end_date: End date (inclusive)
        session: Configured requests session
        
    Returns:
        List of URLs within the specified date range
    """
    logger.info(f"Searching for tracks from {start_date.date()} to {end_date.date()}")
    
    # Strategy 1: Parse existing tracks from profile
    profile_tracks = fetch_profile_tracks(profile_url, session)
    urls_found = []
    
    for url in profile_tracks:
        track_date = extract_date_from_url(url)
        if track_date and start_date <= track_date <= end_date:
            urls_found.append(url)
            logger.info(f"Found matching URL (from profile): {url}")
    
    # Strategy 2: Generate and check potential URLs
    logger.info("Checking for additional URLs using date patterns...")
    username = extract_username_from_url(profile_url)
    
    current_date = start_date
    while current_date <= end_date:
        potential_urls = generate_url_patterns(username, current_date)
        
        for url in potential_urls:
            if url not in urls_found and check_url_exists(url, session):
                urls_found.append(url)
                logger.info(f"Found additional URL (from pattern): {url}")
                time.sleep(0.5)  # Rate limiting
        
        current_date += timedelta(days=1)
    
    return list(dict.fromkeys(urls_found)) # Ensure uniqueness


# ============================================================================
# Audio Download
# ============================================================================

def get_yt_dlp_options(output_dir: Path, audio_quality: str = "192") -> dict:
    """
    Get yt-dlp configuration options.
    
    Args:
        output_dir: Directory to save downloaded files
        audio_quality: Audio quality in kbps (default: 192)
        
    Returns:
        Dictionary of yt-dlp options
    """
    return {
        'format': 'bestaudio/best',
        'outtmpl': str(output_dir / '%(title)s.%(ext)s'),
        'noplaylist': True,
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': audio_quality,
        }],
        'quiet': True,
        'no_warnings': True,
        'ignoreerrors': True, # Continue on 404s or other download errors
        'overwrites': False,
        'download_archive': str(PROJECT_ROOT / 'data' / '.ytdlp-archive.txt'),
    }


def download_single_track(url: str, output_dir: Path, audio_quality: str = "192") -> Optional[str]:
    """
    Download a single SoundCloud track.
    
    Args:
        url: SoundCloud URL to download
        output_dir: Directory to save the audio file
        audio_quality: Audio quality in kbps
        
    Returns:
        Path to downloaded MP3 file if successful, None otherwise
    """
    if not validate_soundcloud_url(url):
        logger.error(f"Invalid SoundCloud URL: {url}")
        return None
    
    ydl_opts = get_yt_dlp_options(output_dir, audio_quality)
    
    # Pre-check expected filename to skip re-downloads
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info_probe = ydl.extract_info(url, download=False)
            if info_probe:
                expected = ydl.prepare_filename(info_probe)
                mp3_expected = f"{os.path.splitext(expected)[0]}.mp3"
                if Path(mp3_expected).exists():
                    logger.info(f"Skipping (already exists): {mp3_expected}")
                    return mp3_expected
    except Exception as e:
        logger.debug(f"Precheck failed for {url}: {e}")

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            
            # extract_info with download=True returns info dict
            # We must check if 'entries' (playlist) or single video
            # But with 'noplaylist': True, it should be a single info dict
            
            # If 'ignoreerrors' was used, ydl.extract_info might return None
            # or the info dict might lack filename if download failed
            if not info:
                 logger.error(f"Download failed for {url}: No info extracted (likely a 404 or private track).")
                 return None

            filename = ydl.prepare_filename(info)
            if not filename:
                logger.error(f"Download failed for {url}: Could not prepare filename.")
                return None

            base, _ = os.path.splitext(filename)
            mp3_file = f"{base}.mp3"
            
            # Check if the file was actually created
            if not Path(mp3_file).exists():
                logger.error(f"Download failed for {url}: MP3 file not created.")
                return None
                
            logger.info(f"Successfully downloaded: {mp3_file}")
            return mp3_file
            
    except Exception as e:
        # This will catch other errors, though yt-dlp with ignoreerrors=True
        # will handle most download-related ones.
        logger.error(f"Download failed for {url}: {str(e)}")
        return None


def download_multiple_tracks(
    urls: List[str], 
    output_dir: Path, 
    audio_quality: str = "192"
) -> Tuple[List[str], List[str]]:
    """
    Download multiple SoundCloud tracks.
    
    Args:
        urls: List of SoundCloud URLs to download
        output_dir: Directory to save audio files
        audio_quality: Audio quality in kbps
        
    Returns:
        Tuple of (successful_downloads, failed_urls)
    """
    logger.info(f"Starting download of {len(urls)} tracks to: {output_dir}")
    
    downloaded_files = []
    failed_urls = []
    
    for i, url in enumerate(urls, 1):
        logger.info(f"Downloading {i}/{len(urls)}: {url}")
        
        result = download_single_track(url, output_dir, audio_quality)
        if result:
            downloaded_files.append(result)
        else:
            failed_urls.append(url)
        
        # Rate limiting
        time.sleep(1)
    
    # Log summary
    logger.info(f"\n{'='*60}")
    logger.info(f"Download Summary:")
    logger.info(f"  Total URLs: {len(urls)}")
    logger.info(f"  Successful: {len(downloaded_files)}")
    logger.info(f"  Failed: {len(failed_urls)}")
    logger.info(f"{'='*60}")
    
    if failed_urls:
        logger.warning("Failed URLs:")
        for url in failed_urls:
            logger.warning(f"  - {url}")
    
    return downloaded_files, failed_urls


# ============================================================================
# Main Download Class
# ============================================================================

class SoundCloudDownloader:
    """
    SoundCloud track downloader with date filtering capabilities.
    
    Handles URL validation, audio extraction, and batch downloads with
    proper error handling and logging. Downloads are saved to the project's
    data/01_raw/ directory.
    """
    
    def __init__(self, audio_quality: str = "192"):
        """
        Initialize the SoundCloud downloader. (FIX 2)
        
        Args:
            audio_quality: Audio quality for MP3 conversion in kbps (default: 192)
        """
        self.audio_quality = audio_quality
        self.raw_data_dir = get_raw_data_dir()
        self.session = create_session()
        self._logged_output_dir = False # Flag to log output dir just once
        
        logger.info(f"Initialized downloader. Base data directory: {self.raw_data_dir}")
    
    def create_output_directory(self, dir_name: str) -> Path:
        """
        Get the /01_raw directory, ensuring it exists. (FIX 2)
        
        Per user request, this method now ignores the 'dir_name'
        and always returns the root '01_raw' directory.
        
        Args:
            dir_name: This argument is ignored.
            
        Returns:
            Path to data/01_raw/ directory
        """
        # dir_name is ignored to place all files directly in /01_raw
        output_dir = self.raw_data_dir
        output_dir.mkdir(parents=True, exist_ok=True)
        
        # Log this once to avoid repetition
        if not self._logged_output_dir:
            logger.info(f"All files will be saved directly to: {output_dir}")
            self._logged_output_dir = True
            
        return output_dir
    
    def download_by_date_range(
        self, 
        profile_url: str, 
        start_date: str, 
        end_date: str,
        output_dir_name: Optional[str] = None
    ) -> List[str]:
        """
        Download SoundCloud tracks within a specified date range.
        
        Args:
            profile_url: SoundCloud profile URL
            start_date: Start date in 'YYYY-MM-DD' format
            end_date: End date in 'YYYY-MM-DD' format
            output_dir_name: Custom output directory name (optional, but ignored)
            
        Returns:
            List of successfully downloaded file paths
            
        Raises:
            ValueError: If date format is invalid
        """
        # Parse dates
        try:
            start_dt = datetime.strptime(start_date, '%Y-%m-%d')
            end_dt = datetime.strptime(end_date, '%Y-%m-%d')
        except ValueError as e:
            raise ValueError(f"Invalid date format. Use YYYY-MM-DD: {e}")
        
        # Create output directory
        # Note: output_dir_name is passed but will be ignored by the
        # modified create_output_directory method.
        if not output_dir_name:
            output_dir_name = f"soundcloud_{start_date}_to_{end_date}"
        output_dir = self.create_output_directory(output_dir_name)
        
        # Collect URLs
        urls = collect_urls_in_date_range(profile_url, start_dt, end_dt, self.session)
        
        if not urls:
            logger.warning("No tracks found in the specified date range")
            return []
        
        # Download tracks
        downloaded_files, _ = download_multiple_tracks(urls, output_dir, self.audio_quality)
        return downloaded_files
    
    def download_urls(
        self, 
        urls: List[str], 
        output_dir_name: Optional[str] = None
    ) -> List[str]:
        """
        Download specific SoundCloud URLs.
        
        Args:
            urls: List of SoundCloud URLs to download
            output_dir_name: Custom output directory name (optional, but ignored)
            
        Returns:
            List of successfully downloaded file paths
        """
        # Note: output_dir_name is passed but will be ignored by the
        # modified create_output_directory method.
        if not output_dir_name:
            timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
            output_dir_name = f"soundcloud_downloads_{timestamp}"
        
        output_dir = self.create_output_directory(output_dir_name)
        
        # Validate URLs before passing them to the downloader
        valid_urls = []
        for url in urls:
            if validate_soundcloud_url(url):
                valid_urls.append(url)
            else:
                logger.warning(f"Skipping invalid URL: {url}")

        if not valid_urls:
            logger.warning("No valid URLs to download.")
            return []

        downloaded_files, _ = download_multiple_tracks(valid_urls, output_dir, self.audio_quality)
        return downloaded_files


# ============================================================================
# Convenience Functions
# ============================================================================

def download_by_date_range(
    profile_url: str, 
    start_date: str, 
    end_date: str,
    output_dir: Optional[str] = None,
    audio_quality: str = "192"
) -> List[str]:
    """
    Download SoundCloud tracks by date range.
    
    Convenience function that creates a downloader instance and downloads
    tracks within the specified date range. Files are saved directly to
    data/01_raw/.
    
    Args:
        profile_url: SoundCloud profile URL
        start_date: Start date in 'YYYY-MM-DD' format
        end_date: End date in 'YYYY-MM-DD' format
        output_dir: Custom output directory name (optional, but ignored)
        audio_quality: Audio quality in kbps (default: 192)
        
    Returns:
        List of successfully downloaded file paths
        
    Example:
        urls = download_by_date_range(
            "https://soundcloud.com/radio-ergo",
            "2025-03-15",
            "2025-03-16"
        )
    """
    downloader = SoundCloudDownloader(audio_quality=audio_quality)
    return downloader.download_by_date_range(profile_url, start_date, end_date, output_dir)


def download_urls(
    urls: List[str], 
    output_dir: Optional[str] = None,
    audio_quality: str = "192"
) -> List[str]:
    """
    Download specific SoundCloud URLs.
    
    Convenience function that creates a downloader instance and downloads
    the specified URLs. Files are saved directly to data/01_raw/.
    
    Args:
        urls: List of SoundCloud URLs to download
        output_dir: Custom output directory name (optional, but ignored)
        audio_quality: Audio quality in kbps (default: 192)
        
    Returns:
        List of successfully downloaded file paths
        
    Example:
        urls = [
            "https://soundcloud.com/radio-ergo/idaacadda-09-mar-2025",
            "https://soundcloud.com/radio-ergo/idaacadda-10-mar-2025"
        ]
        files = download_urls(urls, "radio_ergo_march") # "radio_ergo_march" is ignored
    """
    downloader = SoundCloudDownloader(audio_quality=audio_quality)
    return downloader.download_urls(urls, output_dir)


def download_radio_ergo_by_date(
    start_date: str, 
    end_date: str, 
    output_dir: Optional[str] = None
) -> List[str]:
    """
    Download Radio Ergo tracks by date range.
    
    Specialized convenience function for Radio Ergo downloads.
    
    Args:
        start_date: Start date in 'YYYY-MM-DD' format
        end_date: End date in 'YYYY-MM-DD' format
        output_dir: Custom output directory name (optional, but ignored)
        
    Returns:
        List of successfully downloaded file paths
        
    Example:
        files = download_radio_ergo_by_date("2025-03-01", "2025-03-31")
    """
    return download_by_date_range(
        "https://soundcloud.com/radio-ergo",
        start_date,
        end_date,
        output_dir
    )


# ============================================================================
# Usage Examples
# ============================================================================

# Example 1: Download Radio Ergo tracks for a date range
# files = download_radio_ergo_by_date("2025-03-15", "2025-03-16")

# Example 2: Download specific URLs
# urls = [
#     "https://soundcloud.com/radio-ergo/idaacadda-09-mar-2025",
#     "https://soundcloud.com/radio-ergo/idaacadda-10-mar-2025",
#     "https://soundcloud.com/radio-ergo/not-a-real-link-123" # This will be skipped
# ]
# files = download_urls(urls, "radio_ergo_specific_dates") # This argument is ignored
# All files will be saved to data/01_raw/

2025-11-10 12:24:37,248 - INFO - Logging configured. Log file will be saved to: /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/logs/soundcloud_downloader.log


In [4]:
files = download_radio_ergo_by_date("2022-01-01", "2022-03-31")

2025-11-10 12:24:40,741 - INFO - Initialized downloader. Base data directory: /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw
2025-11-10 12:24:40,744 - INFO - All files will be saved directly to: /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw
2025-11-10 12:24:40,744 - INFO - Searching for tracks from 2022-01-01 to 2022-03-31


2025-11-10 12:24:41,265 - INFO - Checking for additional URLs using date patterns...
2025-11-10 12:24:41,447 - INFO - Found additional URL (from pattern): https://soundcloud.com/radio-ergo/idaacadda-01-jan-2022
2025-11-10 12:24:43,584 - INFO - Found additional URL (from pattern): https://soundcloud.com/radio-ergo/idaacadda-02-jan-2022
2025-11-10 12:24:47,246 - INFO - Found additional URL (from pattern): https://soundcloud.com/radio-ergo/idaacadda-04-jan-2022
2025-11-10 12:24:49,296 - INFO - Found additional URL (from pattern): https://soundcloud.com/radio-ergo/idaacadda-05-jan-2022
2025-11-10 12:24:51,503 - INFO - Found additional URL (from pattern): https://soundcloud.com/radio-ergo/idaacadda-06-jan-2022
2025-11-10 12:24:55,773 - INFO - Found additional URL (from pattern): https://soundcloud.com/radio-ergo/idaacadda-08-jan-2022
2025-11-10 12:24:58,101 - INFO - Found additional URL (from pattern): https://soundcloud.com/radio-ergo/idaacadda-09-jan-2022
2025-11-10 12:24:59,970 - INFO - 